# Análisis de datos de la red de tiendas - RetailNow

Este proyecto analiza datos de ventas, inventarios y satisfacción del cliente de una cadena de tiendas minoristas utilizando **Pandas** para manipulación de datos y **NumPy** para cálculos estadísticos y simulaciones.

Objetivos:
- Analizar ventas por tienda
- Evaluar rotación de inventarios
- Identificar tiendas con inventarios críticos
- Analizar satisfacción del cliente
- Calcular estadísticas con NumPy
- Simular proyecciones de ventas futuras

Importar librerías

In [27]:
# Importación de librerías necesarias
import pandas as pd
import numpy as np

Cargar datos

In [65]:
# Cargar archivos CSV desde /workspace
sales = pd.read_csv("/workspace/sales.csv")
inventories = pd.read_csv("/workspace/inventories.csv")
satisfaction = pd.read_csv("/workspace/satisfaction.csv")

print("Datos de ventas")
display(sales.head())

print("Datos de inventarios")
display(inventories.head())

print("Datos de satisfacción")
display(satisfaction.head())

Datos de ventas


,ID_Tienda,Producto,Cantidad_Vendida,Precio_Unitario,Fecha_Venta
0,1,Producto A,20,100,2023-01-05
1,1,Producto B,15,200,2023-01-06
2,2,Producto A,30,100,2023-01-07
3,2,Producto C,25,300,2023-01-08
4,3,Producto A,10,100,2023-01-09


Datos de inventarios


,ID_Tienda,Producto,Stock_Disponible,Fecha_Actualización
0,1,Producto A,50,2023-01-05
1,1,Producto B,40,2023-01-06
2,2,Producto A,60,2023-01-07
3,2,Producto C,45,2023-01-08
4,3,Producto A,30,2023-01-09


Datos de satisfacción


,ID_Tienda,Satisfacción_Promedio,Fecha_Evaluación
0,1,85,2023-01-15
1,2,90,2023-01-15
2,3,70,2023-01-15
3,4,65,2023-01-15
4,5,55,2023-01-15


Revisar valores nulos

In [66]:
print("Valores nulos en ventas")
print(sales.isnull().sum())

print("Valores nulos en inventarios")
print(inventories.isnull().sum())

print("Valores nulos en satisfacción")
print(satisfaction.isnull().sum())

Valores nulos en ventas
ID_Tienda           0
Producto            0
Cantidad_Vendida    0
Precio_Unitario     0
Fecha_Venta         0
dtype: int64
Valores nulos en inventarios
ID_Tienda              0
Producto               0
Stock_Disponible       0
Fecha_Actualización    0
dtype: int64
Valores nulos en satisfacción
ID_Tienda                0
Satisfacción_Promedio    0
Fecha_Evaluación         0
dtype: int64


Limpieza de datos

In [67]:
sales = sales.dropna()
inventories = inventories.dropna()
satisfaction = satisfaction.dropna()

print("Dimensiones después de limpiar datos")

print("Ventas:", sales.shape)
print("Inventarios:", inventories.shape)
print("Satisfacción:", satisfaction.shape)

Dimensiones después de limpiar datos
Ventas: (10, 5)
Inventarios: (10, 4)
Satisfacción: (5, 3)


Normalizar nombres de columnas

In [68]:
sales = sales.rename(columns={
    "ID_Tienda": "store",
    "Producto": "product",
    "Cantidad_Vendida": "quantity",
    "Precio_Unitario": "price"
})

inventories = inventories.rename(columns={
    "Producto": "product",
    "Stock": "stock"
})

satisfaction = satisfaction.rename(columns={
    "ID_Tienda": "store",
    "Satisfaccion_Promedio": "satisfaction_score"
})

print("Columnas de ventas:")
print(sales.columns)

Columnas de ventas:
Index(['store', 'product', 'quantity', 'price', 'Fecha_Venta'], dtype='object')


Ventas totales por tienda

In [69]:
ventas_por_tienda = sales.groupby("store")["quantity"].sum()

print("Ventas totales por tienda")
print(ventas_por_tienda)

Ventas totales por tienda
store
1    35
2    55
3    50
4    60
5    50
Name: quantity, dtype: int64


Ventas por producto y tienda

In [70]:
ventas_producto_tienda = sales.groupby(["store","product"])["quantity"].sum()

print("Ventas por producto y tienda")
print(ventas_producto_tienda)

Ventas por producto y tienda
store  product   
1      Producto A    20
       Producto B    15
2      Producto A    30
       Producto C    25
3      Producto A    10
       Producto B    40
4      Producto A    25
       Producto C    35
5      Producto B    20
       Producto C    30
Name: quantity, dtype: int64


Calcular ingresos totales

In [71]:
sales["Total_Ventas"] = sales["quantity"] * sales["price"]

ingresos_por_tienda = sales.groupby("store")["Total_Ventas"].sum()

print("Ingresos totales por tienda")
print(ingresos_por_tienda)

Ingresos totales por tienda
store
1     5000
2    10500
3     9000
4    13000
5    13000
Name: Total_Ventas, dtype: int64


Resumen estadístico de ventas

In [72]:
print("Resumen estadístico de ventas")
print(sales["Total_Ventas"].describe())

Resumen estadístico de ventas
count       10.000000
mean      5050.000000
std       3361.960407
min       1000.000000
25%       2625.000000
50%       3500.000000
75%       7875.000000
max      10500.000000
Name: Total_Ventas, dtype: float64


Análisis de inventarios (POR TIENDA)

In [73]:
ventas_tienda_producto = sales.groupby(["store","product"])["quantity"].sum().reset_index()

Unir ventas con inventarios

In [74]:
inventarios_ventas = pd.merge(inventories, ventas_tienda_producto, on="product")

Calcular rotación de inventario

In [76]:
# Rename Stock_Disponible to stock for consistency
inventarios_ventas = inventarios_ventas.rename(columns={"Stock_Disponible": "stock"})

inventarios_ventas["rotacion"] = inventarios_ventas["quantity"] / inventarios_ventas["stock"]
inventarios_ventas["porcentaje_vendido"] = inventarios_ventas["rotacion"] * 100

print("Inventarios con rotación por tienda:")
display(inventarios_ventas.head())

Inventarios con rotación por tienda:


,ID_Tienda,product,stock,Fecha_Actualización,store,quantity,rotacion,porcentaje_vendido
0,1,Producto A,50,2023-01-05,1,20,0.400,40.0
1,1,Producto A,50,2023-01-05,2,30,0.600,60.0
2,1,Producto A,50,2023-01-05,3,10,0.200,20.0
3,1,Producto A,50,2023-01-05,4,25,0.500,50.0
4,1,Producto B,40,2023-01-06,1,15,0.375,37.5


Porcentaje vendido

In [77]:
inventarios_ventas["porcentaje_vendido"] = (inventarios_ventas["quantity"] / inventarios_ventas["stock"]) * 100

print("Inventarios con rotación")
display(inventarios_ventas.head())

Inventarios con rotación


,ID_Tienda,product,stock,Fecha_Actualización,store,quantity,rotacion,porcentaje_vendido
0,1,Producto A,50,2023-01-05,1,20,0.400,40.0
1,1,Producto A,50,2023-01-05,2,30,0.600,60.0
2,1,Producto A,50,2023-01-05,3,10,0.200,20.0
3,1,Producto A,50,2023-01-05,4,25,0.500,50.0
4,1,Producto B,40,2023-01-06,1,15,0.375,37.5


Inventarios críticos (<10%)

In [78]:
inventario_critico = inventarios_ventas[inventarios_ventas["rotacion"] < 0.10]

print("Inventarios críticos")
display(inventario_critico)

Inventarios críticos


,ID_Tienda,product,stock,Fecha_Actualización,store,quantity,rotacion,porcentaje_vendido


Satisfacción del cliente

In [79]:
print("Datos de satisfacción")
display(satisfaction.head())

Datos de satisfacción


,store,Satisfacción_Promedio,Fecha_Evaluación
0,1,85,2023-01-15
1,2,90,2023-01-15
2,3,70,2023-01-15
3,4,65,2023-01-15
4,5,55,2023-01-15


Tiendas con satisfacción menor a 60%

In [81]:
baja_satisfaccion = satisfaction[satisfaction["Satisfacción_Promedio"] < 60]

print("Tiendas con satisfacción menor al 60%")
display(baja_satisfaccion)

Tiendas con satisfacción menor al 60%


,store,Satisfacción_Promedio,Fecha_Evaluación
4,5,55,2023-01-15


Convertir ventas a NumPy

In [82]:
ventas_array = sales["Total_Ventas"].to_numpy()

Mediana de ventas

In [83]:
mediana_ventas = np.median(ventas_array)

print("Mediana de ventas:", mediana_ventas)

Mediana de ventas: 3500.0


Desviación estándar

In [84]:
desviacion_ventas = np.std(ventas_array)

print("Desviación estándar de ventas:", desviacion_ventas)

Desviación estándar de ventas: 3189.4356867634124


Simulación de ventas futuras (NumPy)

In [85]:
np.random.seed(42)

proyeccion_ventas = np.random.normal(
    loc=np.mean(ventas_array),
    scale=np.std(ventas_array),
    size=100
)

print("Primeras proyecciones de ventas")
print(proyeccion_ventas[:10])

Primeras proyecciones de ventas
[ 6634.23784573  4609.01490364  7115.76093733  9907.60577603
  4303.18287048  4303.23523392 10086.79771077  7497.68371242
  3552.64163948  6780.46036522]


Estadísticas de la simulación

In [86]:
print("Estadísticas de proyección")

print("Promedio:", np.mean(proyeccion_ventas))
print("Mediana:", np.median(proyeccion_ventas))
print("Desviación:", np.std(proyeccion_ventas))

Estadísticas de proyección
Promedio: 4718.78821147718
Mediana: 4645.081072338637
Desviación: 2882.025680927093


## Conclusiones

- Se calcularon las ventas totales e ingresos por tienda.
- Se identificaron productos con mayor rotación.
- Se detectaron inventarios críticos (<10%).
- Se identificaron tiendas con baja satisfacción del cliente (<60%).
- Se calcularon estadísticas con NumPy (mediana y desviación estándar).
- Se generaron simulaciones de ventas futuras para apoyar decisiones estratégicas.

El análisis permite optimizar la gestión del inventario y mejorar el rendimiento
general de las tiendas RetailNow.